# 00 — Dataset exploration

Contextualize the labeled CT cases before selecting examples. Inventory and selection use the provided reference masks; report this sampling bias explicitly. The supplied Task03 Liver test images have no masks and are excluded.

Array convention: canonical RAS `(X, Y, Z)` with axial slices `[:, :, k]`; spacing is `(sx, sy, sz)` in millimetres.

## 1. Build the inventory
Run `scripts/explore_dataset.py` for a complete CSV and saved context figures. This cell provides the same reusable entry points for interactive exploration.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from psib.config import load_config
from psib.dataset import build_inventory
cfg = load_config(ROOT / 'config.yaml')
settings = cfg['selection']
inventory = build_inventory(cfg['data_dir'], settings['candidate_pool_size'], settings['seed'])
inventory.head()

## 2. Interpret context
This is a seeded candidate-pool subsample (15 cases by default) from the 131 labeled training cases, not the full dataset. The 70 unlabeled test images are excluded. Discuss case count, volume dimensions and physical spacing, CT HU distribution, liver/tumor label counts, tumor burden, and 3D connected lesion counts. Use the generated `results/figures/` plots and inventory CSV as report inputs.

In [ ]:
inventory.describe(include='all')

## 3. Select report examples
Selection is balanced by tumor-volume terciles, seeded for reproducibility, and favors cases with multiple connected tumor components. Slice indices maximize tumor area with a minimum gap. These criteria use the ground-truth masks.

In [ ]:
from psib.dataset import select_cases, select_slices
case_ids = select_cases(inventory, settings['n_cases'], settings['seed'])
case_ids